[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/quickstart/quickstart.ipynb)

# DecimalAI Quickstart

**Get your first traces and see the version-aware loop in 5 minutes.**

This notebook walks you through DecimalAI's core workflow:

1. Instrument an agent with 2 lines of code
2. Generate traces (no LLM API key needed)
3. Change the agent -> automatic manifest versioning
4. See exactly what changed between versions, and which traces it invalidates

> **Runtime -> Run all works with no account and no keys.** The tool functions are
> mocks, and if no DecimalAI key is present the SDK runs in offline mode: it still
> records traces and still builds a manifest for each version of the agent, it just
> doesn't send anything. Steps 1-5 all run that way. The only thing a key adds is
> the hosted dashboard — every cell that needs one says so in its first line.
>
> For framework-specific examples with real LLMs, see the
> [LangChain quickstart](./quickstart_langchain.ipynb) or
> [OpenAI Agents quickstart](./quickstart_openai_agents.ipynb).

## Step 1 — Install & Configure

In [ ]:
# Install the DecimalAI SDK
!pip install -q decimalai

In [ ]:
import os

import decimalai

# A key is OPTIONAL here. This cell looks for one, and if it doesn't find a real
# one it initializes the SDK in offline mode instead of raising: `enabled=False`
# is a real kill switch (no client, no network call), but tracing, tool
# registration and manifest versioning all still run locally — which is the whole
# of Steps 2-5.
#
# To send traces to your own dashboard, get a key at
# https://app.decimal.ai/settings (Settings -> General -> API keys) and either:
#   * add it as a Colab secret named DECIMAL_API_KEY (the key icon, left sidebar),
#   * export DECIMAL_API_KEY before launching Jupyter, or
#   * flip ASK_FOR_KEY to True below and paste it into the hidden prompt.
ASK_FOR_KEY = False

# The literal this notebook used to assign to DECIMAL_API_KEY. Treated as "no key"
# rather than passed through, so a reader who pastes the snippet from the docs and
# forgets to edit it lands in offline mode instead of on a 401.
PLACEHOLDER = "dai_sk_..."


def find_key(*names) -> str:
    """First real key among env vars, Colab secrets, and (opt-in) a prompt."""
    for name in names:
        value = (os.environ.get(name) or "").strip()
        if value and value != PLACEHOLDER:
            return value

    try:
        from google.colab import userdata  # only exists inside Colab
    except ImportError:
        pass
    else:
        for name in names:
            try:
                value = (userdata.get(name) or "").strip()
            except Exception:
                continue  # secret not set, or notebook denied access to it
            if value and value != PLACEHOLDER:
                return value

    if ASK_FOR_KEY:
        import getpass
        # getpass, never input(): a key typed into a cell is a key in the
        # browser history and in the .ipynb you later share.
        return getpass.getpass(f"{names[0]} (input is hidden): ").strip()

    return ""


# DECIMALAI_API_KEY is the alias the CLI also accepts; init() reads both.
API_KEY = find_key("DECIMAL_API_KEY", "DECIMALAI_API_KEY")
ONLINE = False

if API_KEY:
    try:
        decimalai.init(api_key=API_KEY)
        ONLINE = True
    except Exception as exc:
        # init(verify=True) raises on 401/403 or an unreachable backend. An
        # expired key is not a reason to end the notebook in a traceback.
        print(f"!  that key was rejected: {type(exc).__name__}: {exc}")
        print("!  falling back to offline mode — every cell below still runs.\n")

if not ONLINE:
    decimalai.init(enabled=False)

print("ONLINE  — traces will be sent to https://app.decimal.ai/traces"
      if ONLINE else
      "OFFLINE — running without a usable DecimalAI key. Traces and manifests\n"
      "          are built locally and sent nowhere. Steps 2-5 run in full;\n"
      "          only the dashboard links at the end need an account.")

## Step 2 — Define Your Agent (v1)

We'll create a simple customer support agent with two tools:
- `search_docs` — searches a knowledge base
- `check_inventory` — checks product stock levels

These are mock functions (no real API calls), but DecimalAI traces them
the same way it would trace real tools.

`init()` and `@decimalai.trace()` are the two lines that record a run.
`@decimalai.tool` is a third, optional one, and it is what makes versioning work:
it does not change what the function does — it registers the function's name and
argument schema so that **every** trace this agent produces declares the same tool
set, whether or not a given run happened to call every tool. Without it, a manifest
can only describe the tools that fired in that one trace, so an agent that routes
each question to one of two tools mints two alternating manifests instead of one
version.

In [ ]:
# --- Agent v1: Two tools ---

@decimalai.tool
def search_docs(query: str) -> str:
    """Search the knowledge base for relevant articles."""
    responses = {
        "password": "To reset your password, go to Settings > Security > Reset Password.",
        "return": "Our return policy allows returns within 30 days of purchase.",
        "shipping": "Standard shipping takes 5-7 business days. Express: 1-2 days.",
        "support": "Contact support at help@example.com or call 1-800-555-0123.",
    }
    for keyword, response in responses.items():
        if keyword in query.lower():
            return response
    return f"Found 3 articles related to '{query}'. Please be more specific."


@decimalai.tool
def check_inventory(product_id: str) -> dict:
    """Check product stock levels."""
    inventory = {
        "SKU-1234": {"product_id": "SKU-1234", "name": "Wireless Mouse", "in_stock": True, "quantity": 42},
        "SKU-5678": {"product_id": "SKU-5678", "name": "USB-C Hub", "in_stock": False, "quantity": 0},
    }
    return inventory.get(product_id, {"product_id": product_id, "in_stock": True, "quantity": 100})


from decimalai.decorators import get_registered_tools

print("Agent v1 declares:", ", ".join(sorted(t["name"] for t in get_registered_tools())))

## Step 3 — Instrument & Generate Traces

The `@decimalai.trace()` decorator captures everything — inputs, outputs,
tool calls, and timing. With a key it ships that to your dashboard; without one
it still builds the trace and the manifest, and drops them at the network edge.

Each query below routes to **one** tool, so the traces differ from each other —
that is the point, and it is what makes the impact analysis in Step 5 non-trivial.
The *manifest* is the same for all six, because the tool registry, not the
individual run, is what declares the agent's surface.

In [ ]:
from decimalai.schema.manifest import extract_from_config
from decimalai.skills import discover_skills

# The six questions. Both versions of the agent answer the same six, so the only
# thing that differs between v1 and v2 is the agent — not the workload.
QUERIES = [
    "How do I reset my password?",
    "Check inventory for SKU-1234",
    "What is your return policy?",
    "Is product SKU-5678 in stock?",
    "How do I contact support?",
    "I need a refund for a damaged item",
]

# Which tool each query actually reached, kept so Step 5 can count the traces
# that a version change strands. The backend keeps this per trace; offline we
# have to keep it ourselves.
TOOL_USED = {"v1": {}, "v2": {}}


@decimalai.trace(agent_name="support-agent")
def run_agent_v1(query: str) -> str:
    """Simple support agent that routes to the right tool."""
    if "stock" in query.lower() or "inventory" in query.lower():
        result = check_inventory("SKU-1234")
        decimalai.log_tool_call(name="check_inventory", input={"product_id": "SKU-1234"}, output=result)
        TOOL_USED["v1"][query] = "check_inventory"
        return f"Stock check: {result['name']} — {'In stock' if result['in_stock'] else 'Out of stock'} ({result['quantity']} units)"
    else:
        result = search_docs(query)
        decimalai.log_tool_call(name="search_docs", input={"query": query}, output=result)
        TOOL_USED["v1"][query] = "search_docs"
        return f"Here's what I found: {result}"


def manifest_now(agent_name="support-agent"):
    """The manifest snapshot the tracer builds at the end of a trace.

    Same call the SDK makes internally (decimalai/generic.py), over the same
    inputs: the @decimalai.tool registry plus any skills found on disk. Recomputed
    here so the diff in Step 5 works with no account.
    """
    return extract_from_config(
        agent_name=agent_name,
        tools=get_registered_tools(),
        skills=discover_skills() or None,
    )


print(f"Running {len(QUERIES)} queries through Agent v1...\n")
for q in QUERIES:
    answer = run_agent_v1(q)
    print(f"  Q: {q}")
    print(f"  A: {answer}")
    print(f"     [tool: {TOOL_USED['v1'][q]}]\n")

MANIFEST_V1 = manifest_now()
V1_TOOLS = sorted(c.component_name for c in MANIFEST_V1.components
                  if c.component_type == "tool")

print(f"{len(QUERIES)} traces recorded" + (" and sent to DecimalAI." if ONLINE
                                           else " locally (offline: nothing sent)."))
print(f"manifest v1  {MANIFEST_V1.manifest_hash[:12]}  tools: {', '.join(V1_TOOLS)}")
print("Every one of those traces reports the same manifest — one version, not one per branch.")

## Step 4 — Update the Agent (v2)

Now let's simulate a real-world agent update:

1. **Rename** `check_inventory` -> `lookup_stock` (clearer name)
2. **Add** a new tool: `process_refund`

This is exactly what happens when your team iterates on an agent.

One notebook-specific wrinkle: in production, v2 is a **new process** — the old
code is gone and the tool registry starts empty. A notebook keeps one Python
process alive across both versions, so the cell below clears the registry first.
Without that, `check_inventory` would still be registered, v2's manifest would
report four tools instead of three, and the rename would look like an addition.

In [ ]:
# --- Agent v2: Renamed tool + new tool ---

from decimalai.decorators import _registered_tools

# Stand in for the process restart that a real deploy is. The registry is a
# module-level dict that only ever grows within a process; a tool that is deleted
# from your source is only "gone" once the process that registered it is.
_registered_tools.clear()

# Unchanged in v2 — re-registered because the line above cleared everything.
search_docs = decimalai.tool(search_docs)


@decimalai.tool
def lookup_stock(item_id: str) -> dict:
    """Look up current stock for an item. (Renamed from check_inventory)"""
    inventory = {
        "SKU-1234": {"item_id": "SKU-1234", "name": "Wireless Mouse", "in_stock": True, "quantity": 42},
        "SKU-5678": {"item_id": "SKU-5678", "name": "USB-C Hub", "in_stock": False, "quantity": 0},
    }
    return inventory.get(item_id, {"item_id": item_id, "in_stock": True, "quantity": 100})


@decimalai.tool
def process_refund(order_id: str, reason: str) -> dict:
    """Process a refund for an order. (NEW in v2)"""
    return {"order_id": order_id, "status": "refunded", "reason": reason, "amount": 29.99}


print("Agent v2 declares:", ", ".join(sorted(t["name"] for t in get_registered_tools())))
print("   - search_docs    unchanged")
print("   - lookup_stock   renamed from check_inventory")
print("   - process_refund new")

In [ ]:
@decimalai.trace(agent_name="support-agent")
def run_agent_v2(query: str) -> str:
    """Updated support agent with renamed + new tools."""
    if "stock" in query.lower() or "inventory" in query.lower():
        result = lookup_stock("SKU-1234")
        decimalai.log_tool_call(name="lookup_stock", input={"item_id": "SKU-1234"}, output=result)
        TOOL_USED["v2"][query] = "lookup_stock"
        return f"Stock check: {result['name']} — {'In stock' if result['in_stock'] else 'Out of stock'}"
    elif "refund" in query.lower():
        result = process_refund("ORD-001", "damaged item")
        decimalai.log_tool_call(name="process_refund", input={"order_id": "ORD-001"}, output=result)
        TOOL_USED["v2"][query] = "process_refund"
        return f"Refund processed: ${result['amount']} for order {result['order_id']}"
    else:
        result = search_docs(query)
        decimalai.log_tool_call(name="search_docs", input={"query": query}, output=result)
        TOOL_USED["v2"][query] = "search_docs"
        return f"Here's what I found: {result}"


# The SAME six queries. Re-running the workload is what makes the two versions
# comparable: any difference below is the agent changing, not the questions.
print(f"Running the same {len(QUERIES)} queries through Agent v2...\n")
for q in QUERIES:
    answer = run_agent_v2(q)
    print(f"  Q: {q}")
    print(f"  A: {answer}")
    print(f"     [tool: {TOOL_USED['v1'][q]} -> {TOOL_USED['v2'][q]}]\n")

MANIFEST_V2 = manifest_now()
V2_TOOLS = sorted(c.component_name for c in MANIFEST_V2.components
                  if c.component_type == "tool")

if ONLINE:
    decimalai.flush()  # drain the background sender before you go look

print(f"manifest v2  {MANIFEST_V2.manifest_hash[:12]}  tools: {', '.join(V2_TOOLS)}")
print("changed:", MANIFEST_V1.manifest_hash != MANIFEST_V2.manifest_hash)

## Step 5 — What Changed, and What It Costs You

Two manifests, one diff. This runs with no account: the hashes and the tool sets
below were computed by the same extractor the tracer uses, from the same registry
the tracer reads.

In [ ]:
v1, v2 = set(V1_TOOLS), set(V2_TOOLS)
removed, added, kept = sorted(v1 - v2), sorted(v2 - v1), sorted(v1 & v2)

print(f"manifest v1  {MANIFEST_V1.manifest_hash[:12]}  {', '.join(sorted(v1))}")
print(f"manifest v2  {MANIFEST_V2.manifest_hash[:12]}  {', '.join(sorted(v2))}\n")
for label, names in (("gone   ", removed), ("new    ", added), ("kept   ", kept)):
    for n in names:
        print(f"  {label} {n}")

# Which of the v1 traces are stranded by the change. A trace is stranded when it
# called a tool the new manifest no longer declares: replayed against v2 it would
# ask for something that does not exist, and used as training data it would teach
# the model to do the same.
stranded = [q for q, t in TOOL_USED["v1"].items() if t in removed]

print(f"\n{len(stranded)} of {len(QUERIES)} v1 traces called a tool that v2 removed:")
for q in stranded:
    print(f"  - {q!r}  (called {TOOL_USED['v1'][q]})")
print(f"The other {len(QUERIES) - len(stranded)} only used tools that survived the change.")

print(f"\nAnd the same {len(QUERIES)} questions now route differently:")
for q in QUERIES:
    if TOOL_USED["v1"][q] != TOOL_USED["v2"][q]:
        print(f"  {TOOL_USED['v1'][q]:<16} -> {TOOL_USED['v2'][q]:<16} {q!r}")

### The same thing, with an account

**Needs a key.** With one, the six traces above are stored against their manifest,
and **[your dashboard](https://app.decimal.ai)** -> **Agents -> support-agent**
turns the diff you just printed into an **Impact Report** over every trace you have
ever recorded for this agent — not only the six in this session:

| Classification | What it means |
|---|---|
| **Keep** | The trace only touched parts of the agent the change didn't affect. |
| **Repair** | The trace references something that moved but has a mechanical equivalent — e.g. a rename that can be rewritten in place. |
| **Replay** | The change is behavioural; the trace has to be re-run before it can be trusted. |
| **Drop** | What the trace used is gone with no equivalent. |

### Why This Matters

The cell above named the traces that called `check_inventory`. If you fine-tuned on
your raw trace history today, those rows would teach your model to call a tool that
no longer exists. The manifest is what makes that question answerable at all: without
one, a trace is just text that looks fine.

## What Just Happened

```
Agent v1 answers 6 questions -> 6 traces, all reporting ONE manifest
    |
Agent changes (tool renamed, tool added)
    |
Agent v2 answers the SAME 6 questions -> a second manifest, auto-detected
    |
Diff the two -> which tools moved, and which stored traces are stranded
    |
With an account: keep / repair / replay / drop over your whole trace history,
then repair the stale ones and build a clean dataset
```

This happens **automatically, every time your agent changes** — the only thing you
wrote was `@decimalai.tool` and `@decimalai.trace()`.

## Next Steps

- [LangChain Quickstart](./quickstart_langchain.ipynb) — Instrument a real LangChain agent
- [OpenAI Agents Quickstart](./quickstart_openai_agents.ipynb) — Instrument an OpenAI Agents app
- [Concepts](https://docs.decimal.ai/concepts) — Understand manifests, traces, and datasets
- [Dashboard](https://app.decimal.ai) — Explore your traces (needs a free account)